
# 1) Data Collection

Tennis Grand Slam Knowledge Graph and Intelligent QA System (RAG) 

Step 1: Web Crawling and Information Extraction Preparation


A knowledge graph is only as good as the source data used to build it. If we crawl pages that are noisy, too short, or unrelated to our domain, the next steps will extract poor entities and poor relations. This creates weak triples, alignment errors, and low-quality answers in the final RAG system.

For this reason, the first notebook does not only download pages. It builds a **clean, traceable, and ethical collection pipeline**. Every document we keep must have a source URL, a title, cleaned text, and a word count. This traceability will help us justify our extraction decisions later in the project.



## Seed URL Strategy
We start from a small set of Wikipedia pages because they are:
- text-rich;
- strongly related to the Grand Slam domain;
- relatively stable and well structured;
- easy to clean with content extraction tools.

The list mixes:
- general pages about Grand Slam tournaments;
- tournament pages for the four major events;
- selected edition pages for concrete match-level evidence;
- player pages for winners, opponents, nationality, and career context.

In [1]:
from __future__ import annotations

import json
import logging
import sys
from pathlib import Path

import pandas as pd
import requests

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.crawl.wikipedia_crawler import (
    configure_logging,
    crawl_urls,
    load_jsonl,
    save_jsonl,
)

In [2]:
configure_logging(logging.INFO)

DATA_DIR = PROJECT_ROOT / "data" / "raw"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = DATA_DIR / "crawler_output.jsonl"

SEED_URLS = [
    "https://en.wikipedia.org/wiki/Grand_Slam_(tennis)",
    "https://en.wikipedia.org/wiki/Australian_Open",
    "https://en.wikipedia.org/wiki/French_Open",
    "https://en.wikipedia.org/wiki/Wimbledon_Championships",
    "https://en.wikipedia.org/wiki/US_Open_(tennis)",
    "https://en.wikipedia.org/wiki/2023_Wimbledon_Championships_%E2%80%93_Men%27s_singles",
    "https://en.wikipedia.org/wiki/2022_French_Open_%E2%80%93_Men%27s_singles",
    "https://en.wikipedia.org/wiki/2021_US_Open_%E2%80%93_Men%27s_singles",
    "https://en.wikipedia.org/wiki/Rafael_Nadal",
    "https://en.wikipedia.org/wiki/Novak_Djokovic",
    "https://en.wikipedia.org/wiki/Carlos_Alcaraz",
]

print(f"Project root: {PROJECT_ROOT}")
print(f"Number of seed URLs: {len(SEED_URLS)}")


Project root: /Users/vincentlemeur/Documents/S8/DIA/Datamining/Project_datamining_bis
Number of seed URLs: 11



## Ethical Crawling Discussion
Even for a small academic project, crawling should be respectful.

### 1. Why check `robots.txt`?
The `robots.txt` file tells crawlers which parts of a site are restricted or sensitive. It is not a legal contract by itself, but it is a strong technical and ethical signal. Ignoring it may overload a website or break the website owner's policy.

### 2. Why use delays?
Sending too many requests in a short time can stress the server. We therefore use a short delay between requests. For only a few Wikipedia pages this is simple, but the same principle becomes essential at larger scale.

### 3. Why cleaning matters for ethics and quality
Boilerplate text such as menus, navigation links, cookie banners, or legal notices adds noise. If we keep that noise, later NLP steps may extract wrong entities such as generic site labels instead of domain facts. Good cleaning protects both the server and the quality of the future KG.


In [3]:
robots_url = "https://en.wikipedia.org/robots.txt"
robots_response = requests.get(robots_url, timeout=20)
print(f"robots.txt status: {robots_response.status_code}")
print("First 20 lines of robots.txt:\n")
print("\n".join(robots_response.text.splitlines()[:20]))


robots.txt status: 403
First 20 lines of robots.txt:

Please set a user-agent and respect our robot policy https://w.wiki/4wJS. See also https://phabricator.wikimedia.org/T400119.



## Collection Pipeline Design
The pipeline below follows four simple rules.

1. Fetch HTML with robust error handling.
2. Extract the main text with `trafilatura` instead of parsing raw HTML manually.
3. Keep only useful pages with at least 500 words.
4. Save each retained page as one JSON object in a JSONL file.

We use `trafilatura` because it is designed to keep the main article content and remove common web boilerplate. This is better than keeping raw HTML, because information extraction models work much better on clean natural-language text than on page menus and markup.


In [4]:
MIN_WORDS = 500
records, crawl_stats = crawl_urls(SEED_URLS, min_words=MIN_WORDS)
save_jsonl(records, str(OUTPUT_PATH))

print("Crawling finished.")
print(json.dumps(crawl_stats, indent=2))
print(f"Saved useful pages to: {OUTPUT_PATH}")

INFO | Fetching https://en.wikipedia.org/wiki/Grand_Slam_(tennis)


INFO | Fetching https://en.wikipedia.org/wiki/Australian_Open


INFO | Fetching https://en.wikipedia.org/wiki/French_Open


INFO | Fetching https://en.wikipedia.org/wiki/Wimbledon_Championships


INFO | Fetching https://en.wikipedia.org/wiki/US_Open_(tennis)


...

INFO | Fetching https://en.wikipedia.org/wiki/2021_US_Open_%E2%80%93_Men%27s_singles


INFO | Fetching https://en.wikipedia.org/wiki/Rafael_Nadal


INFO | Fetching https://en.wikipedia.org/wiki/Novak_Djokovic


INFO | Fetching https://en.wikipedia.org/wiki/Carlos_Alcaraz


Crawling finished.
{
  "fetched": 11,
  "useful": 11,
  "missing_text": 0,
  "filtered_short": 0,
  "request_failed": 0,
  "blocked_by_robots": 0
}
Saved useful pages to: /Users/vincentlemeur/Documents/S8/DIA/Datamining/Project_datamining_bis/data/raw/crawler_output.jsonl



## Inspect the Stored JSONL File
JSONL is a practical format for NLP pipelines because:
- each document stays independent;
- the file is easy to stream line by line;
- it preserves metadata such as source URL and title;
- it is easier to audit than a binary format.

Each line must contain at least:
- `url`
- `title`
- `text`
- `word_count`


In [5]:
rows = load_jsonl(str(OUTPUT_PATH))
collection_df = pd.DataFrame(rows)
collection_df[["title", "url", "word_count"]].sort_values("word_count", ascending=False).head(11)

,title,url,word_count
8,Rafael Nadal,https://en.wikipedia.org/wiki/Rafael_Nadal,30221
9,Novak Djokovic,https://en.wikipedia.org/wiki/Novak_Djokovic,29771
10,Carlos Alcaraz,https://en.wikipedia.org/wiki/Carlos_Alcaraz,17216
3,Wimbledon Championships,https://en.wikipedia.org/wiki/Wimbledon_Champi...,13670
0,Grand Slam (tennis),https://en.wikipedia.org/wiki/Grand_Slam_(tennis),9685
4,US Open (tennis),https://en.wikipedia.org/wiki/US_Open_(tennis),5656
1,Australian Open,https://en.wikipedia.org/wiki/Australian_Open,5528
2,French Open,https://en.wikipedia.org/wiki/French_Open,5011
6,2022 French Open %E2%80%93 Men%27s singles,https://en.wikipedia.org/wiki/2022_French_Open...,1803
7,2021 US Open %E2%80%93 Men%27s singles,https://en.wikipedia.org/wiki/2021_US_Open_%E2...,1351



## Validation Block
This section is important for academic reproducibility. We do not just trust the crawler. We check:
- how many pages were fetched;
- how many useful pages were kept;
- whether any page had missing text;
- whether short pages were filtered out;
- whether the cleaned text looks readable.

These checks help us detect early problems before moving to information extraction.


In [6]:

validation_summary = {
    "seed_urls": len(SEED_URLS),
    "fetched_pages": crawl_stats["fetched"],
    "useful_pages_retained": crawl_stats["useful"],
    "missing_text_pages": crawl_stats["missing_text"],
    "short_pages_filtered": crawl_stats["filtered_short"],
    "request_failures": crawl_stats["request_failed"],
    "minimum_word_threshold": MIN_WORDS,
    "saved_documents": len(rows),
}

print(json.dumps(validation_summary, indent=2))


{
  "seed_urls": 11,
  "fetched_pages": 11,
  "useful_pages_retained": 11,
  "missing_text_pages": 0,
  "short_pages_filtered": 0,
  "request_failures": 0,
  "minimum_word_threshold": 500,
  "saved_documents": 11
}


In [7]:
# Show a few cleaned text snippets to verify that the extraction is readable.
for row in rows[:3]:
    print("=" * 100)
    print(f"TITLE: {row['title']}")
    print(f"URL: {row['url']}")
    print(f"WORD COUNT: {row['word_count']}")
    print("TEXT PREVIEW:")
    print(row['text'][:800])
    print("\n")


TITLE: Grand Slam (tennis)
URL: https://en.wikipedia.org/wiki/Grand_Slam_(tennis)
WORD COUNT: 9685
TEXT PREVIEW:
The Grand Slam in tennis is the achievement of winning all four major championships in one discipline in a calendar year. In doubles, a Grand Slam may be achieved as a team or as an individual with different partners. Winning all four major championships consecutively but not within the same calendar year is referred to as a "non-calendar-year Grand Slam", while winning the four majors at any point during the course of a career is known as a "Career Grand Slam".[1][2] The term Grand Slam is also attributed to the Grand Slam tournaments,[2] referred to as Majors,[3] and they are the world's four most important annual professional tennis tournaments. They offer the most ranking points, prize money,[3] public and media attention, the greatest strength and size of the field and, in recent year


TITLE: Australian Open
URL: https://en.wikipedia.org/wiki/Australian_Open
WORD COUNT